# Лабораторная работа: от порогового нейрона к многослойному персептрону
**Продолжительность: 90 минут.** Работа индивидуально или в парах.

**Цель:** связать формулы лекции с работающим кодом, проверить условия сходимости,
отличить запоминание от обобщения и объяснить роль скрытого слоя.

**До занятия:** Python 3, Jupyter, NumPy, Matplotlib уже установлены; среда проверена.
Достаточно знания функций Python, циклов, массивов и производной сложной функции.
Все данные встроены; скачивать датасеты и устанавливать пакеты во время занятия не нужно.

**Как работать:** запускайте ячейки по порядку; изменяйте места `TODO`; в Markdown-ячейках
«Ответ» пишите кратко. Ячейки с `NotImplementedError` намеренно требуют заполнения.
Готовую инфраструктуру переписывать не нужно. По завершении перезапустите ядро и выполните все ячейки.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 7
np.set_printoptions(precision=3, suppress=True)
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
d_and = np.array([0, 0, 0, 1])
d_xor = np.array([0, 1, 1, 0])
print('Среда готова:', np.__version__)


## A. Биологическая аналогия и математическая модель

1. Проанализируйте сравнение мозга с идеализированной машиной фон Неймана:
   хранение и обработка информации, параллельность, элементы сети.
2. Заполните таблицу: что в нашей модели соответствует дендритам, аксону и силе связи?
3. По оценкам лекции $10^{11}$ нейронов и $10^3$–$10^4$ связей на нейрон вычислите
   порядок числа связей. Почему из него нельзя прямо получить объем памяти в байтах?
4. В электротехнической аналогии сила связи связана с проводимостью (сименсы).
   Есть ли основания присвоить те же единицы каждому весу программной модели?
5. Чем нейросеть как математическая модель отличается от ее программной или аппаратной реализации?


| Элемент | Назначение в аналогии | Соответствие в модели |
|---|---|---|
| Дендриты | … | … |
| Аксон | … | … |
| Сила синаптической связи | … | … |

Число связей: … . Почему это не число байтов: … .
Мозг / машина фон Неймана: … . Нейросеть / нейрокомпьютер: … .


## B. Пороговый нейрон и логические функции

$$s=\sum_j w_jx_j+b,\qquad y=H(s),\qquad H(s)=\begin{cases}1,&s\geq0,\\0,&s<0.\end{cases}$$

Порог $\theta$ связан со смещением: $b=-\theta$. При $s=0$ выход равен **1**.

**B1.** Допишите `step` и `threshold_neuron` (две строки вычислений).
**B2.** Подберите веса и смещения для AND, OR, NOT. Проверьте все строки таблиц истинности.
Используйте формы AND/OR: два веса, NOT: один вес. Нарисуйте схему словами или укажите
входы, веса, сумматор, смещение и выход; график ступеньки строится готовой ячейкой.


In [ ]:
def step(s):
    # TODO: вернуть 0/1 для числа или массива; на границе s=0 вернуть 1
    raise NotImplementedError('Допишите step')

def threshold_neuron(X, w, b):
    # TODO: скалярное произведение / матричное умножение и активация
    raise NotImplementedError('Допишите threshold_neuron')


In [ ]:
# TODO: замените None весами np.array([...]) и числовым смещением
w_and, b_and = None, None
w_or, b_or = None, None
w_not, b_not = None, None


In [ ]:
assert np.array_equal(step(np.array([-1., 0., 1.])), [0, 1, 1])
assert np.array_equal(threshold_neuron(X_logic, w_and, b_and), d_and)
assert np.array_equal(threshold_neuron(X_logic, w_or, b_or), [0, 1, 1, 1])
assert np.array_equal(threshold_neuron(np.array([[0], [1]]), w_not, b_not), [1, 0])
print('Таблицы истинности проверены')

z = np.linspace(-4, 4, 401)
plt.plot(z, step(z), label='H(s)'); plt.scatter([0], [1], s=35)
plt.xlabel('s'); plt.ylabel('y'); plt.title('Пороговая активация')
plt.grid(); plt.legend(); plt.show()


Формулы s и y: … . Связь b и θ: … .
Схема нейрона AND (входы → веса → сумма со смещением → активация → выход): … .
Почему NOT требует отрицательного веса: … .


## C. Обучение, желаемый ответ и сходимость

$d$ — требуемый ответ, $y$ — текущий ответ, $e=d-y$.
Для порогового персептрона используем:
$$w\leftarrow w+\eta e x,\qquad b\leftarrow b+\eta e.$$
В лекции это называется обычным дельта-правилом. В этой работе отличаем его
от градиентного правила для сигмоидного нейрона в блоке E.

**Важно:** эпоха — полный проход по обучающим примерам; обновление выполняется после
отдельного примера. История ниже хранит число неверных ответов **после** полного прохода.
Остановка по лимиту эпох сама по себе не доказывает неразделимость данных.

До запуска вручную рассчитайте одно обновление:
$x=(1,0)$, $d=1$, $w=(-0.2,0.1)$, $b=-0.1$, $\eta=0.5$.


s=…; y=…; e=…; новые w=…; новое b=… .


В готовом цикле допишите только ошибку, изменение весов и смещения.

In [ ]:
def fit_perceptron(X, d, eta=0.1, epochs=200, seed=SEED):
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.1, X.shape[1])
    b = 0.0
    history = []
    for epoch in range(epochs):
        for x, target in zip(X, d):
            y = threshold_neuron(x, w, b)
            # TODO: три строки: e, обновление w, обновление b
            raise NotImplementedError('Допишите обновление персептрона')
        mistakes = int(np.count_nonzero(threshold_neuron(X, w, b) != d))
        history.append(mistakes)
        if mistakes == 0:
            break
    return w, b, np.array(history)


Запустите на AND и XOR. Сопоставьте кривые ошибок с условием линейной разделимости.

In [ ]:
wa, ba, hist_and = fit_perceptron(X_logic, d_and)
wx, bx, hist_xor = fit_perceptron(X_logic, d_xor, epochs=40)
assert np.array_equal(threshold_neuron(X_logic, wa, ba), d_and)
print('AND:', threshold_neuron(X_logic, wa, ba), 'эпох:', len(hist_and))
print('XOR:', threshold_neuron(X_logic, wx, bx), 'ошибок:', hist_xor[-1])
plt.plot(np.arange(1, len(hist_and)+1), hist_and, 'o-', label='AND')
plt.plot(np.arange(1, len(hist_xor)+1), hist_xor, label='XOR')
plt.xlabel('Эпоха'); plt.ylabel('Неверных ответов из 4')
plt.grid(); plt.legend(); plt.show()

# Готовая визуализация: на плоскости входов классы XOR расположены крест-накрест.
plt.scatter(X_logic[:, 0], X_logic[:, 1], c=d_xor, cmap='coolwarm', s=150)
for x, d in zip(X_logic, d_xor): plt.annotate(str(d), x + 0.04)
plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('XOR: одна прямая не разделяет классы')
plt.grid(); plt.show()


- В каких параметрах модель хранит результат обучения? Почему начальные значения можно менять?
- Сформулируйте теорему сходимости с условием разделимости. Почему она не решает XOR?
- Как $\eta$ влияет на величину коррекции? Почему вывод об устойчивости градиентного
  спуска нельзя автоматически переносить на пороговое правило?
- В16 / К14: какая теорема названа в лекции «самой доказанной»? Это цитата из лекции,
  а не условие или часть математического результата.

**Ответ:** …


## D. Пиксельные образы, выходы и обобщение

Рассматриваем цифры из 12 бинарных пикселей (4×3): пиксель изображения дает 1,
фон дает 0. Вектор $x$ — развернутая картинка, а не само числовое значение цифры.
Цель: $d=1$ для четных цифр и $d=0$ для нечетных.

Посмотрите на кодировку, запустите уже написанное обучение.


In [ ]:
digit_patterns = [
    '111101101111', '010110010111', '111001110111', '111011001111',
    '101101111001', '111100011111', '100111101111', '111001010010',
    '111101111111', '111101111001',
]
X_digits = np.array([[int(c) for c in p] for p in digit_patterns], dtype=float)
d_digits = (np.arange(10) % 2 == 0).astype(int)
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for digit, ax in enumerate(axes):
    ax.imshow(X_digits[digit].reshape(4, 3), cmap='binary', vmin=0, vmax=1)
    ax.set_title(str(digit)); ax.axis('off')
plt.show()
wd, bd, hist_digits = fit_perceptron(X_digits, d_digits)
print('Ответы на эталоны:', threshold_neuron(X_digits, wd, bd))
print('Требуемые ответы: ', d_digits)

def distorted_digits(p, seed=101, copies=30):
    X = np.repeat(X_digits, copies, axis=0)
    d = np.repeat(d_digits, copies)
    flips = np.random.default_rng(seed).random(X.shape) < p
    X_test = np.where(flips, 1-X, X)
    # Неизмененные копии исключаем: тестируем действительно новые образы.
    changed = np.any(flips, axis=1)
    return X_test[changed], d[changed]


На новых искаженных картинках вычислите долю правильных ответов.
Эти картинки не участвуют в подборе весов; фиксируем параметры до просмотра теста.

Сравните точность на обучении и при двух уровнях независимых искажений.

In [ ]:
def accuracy(X, d, w, b):
    # TODO: среднее индикаторов совпадения предсказаний с d
    raise NotImplementedError('Вычислите accuracy')

print('Обучение:', accuracy(X_digits, d_digits, wd, bd))
for p in [0.05, 0.20]:
    X_test, d_test = distorted_digits(p)
    print(f'p={p:.2f}; новых изображений={len(d_test)}; accuracy={accuracy(X_test, d_test, wd, bd):.3f}')


Почему 1 выход достаточен для четности, но для схемы one-hot нужно 33 выхода для букв
и 43 для букв и цифр? … . Что хранит вектор d для многоклассового примера? … .
Обучающая / тестовая точность: … . Что они говорят об обобщении? … .
Что нужно изменить в данных и проверке для рукописных букв? … .


## E. Сигмоида, функция ошибки и градиент

Для одного выхода и $N$ примеров:
$$y=\sigma(s)=\frac1{1+e^{-s}},\quad \sigma'(s)=y(1-y),\quad
L=\frac1{2N}\sum_{n=1}^N(y_n-d_n)^2.$$
$$\nabla_w L=X^T\delta,\quad \frac{\partial L}{\partial b}=\sum_n\delta_n,\quad
\delta_n=\frac{(y_n-d_n)y_n(1-y_n)}{N}.$$
$$w\leftarrow w-\eta\nabla_w L,\quad b\leftarrow b-\eta\frac{\partial L}{\partial b}.$$

Здесь δ записана для **вычитания градиента**. В лекции «нейронная ошибка» имеет
противоположный знак: $(d-y)y(1-y)$, и коррекция прибавляется. Оба соглашения эквивалентны.

Реализуйте сигмоиду и половину средней квадратичной ошибки.


In [ ]:
def sigmoid(s):
    s = np.clip(np.asarray(s, dtype=float), -500, 500)
    # TODO: формула логистической функции
    raise NotImplementedError('Допишите sigmoid')

def half_mse(y, d):
    # TODO: половина среднего квадрата разности
    raise NotImplementedError('Допишите half_mse')


Допишите δ и два градиента; проверьте производные численно.

In [ ]:
def sigmoid_gradients(X, d, w, b):
    y = sigmoid(X @ w + b)
    # TODO: δ (для вычитания градиента), dw, db
    raise NotImplementedError('Допишите три вычисления градиента')
    return dw, db


In [ ]:
assert np.allclose(sigmoid(np.array([0.])), [0.5])
assert np.allclose(half_mse(np.array([0., 1.]), np.array([1., 1.])), 0.25)

# Численная проверка знака и множителей в градиенте.
w0, b0 = np.array([0.2, -0.3]), 0.1
dw, db = sigmoid_gradients(X_logic, d_and, w0, b0)
eps = 1e-6
loss_at = lambda w, b: half_mse(sigmoid(X_logic @ w+b), d_and)
numeric_dw = np.empty_like(w0)
for j in range(len(w0)):
    h = np.zeros_like(w0); h[j] = eps
    numeric_dw[j] = (loss_at(w0+h, b0)-loss_at(w0-h, b0))/(2*eps)
numeric_db = (loss_at(w0, b0+eps)-loss_at(w0, b0-eps))/(2*eps)
assert np.allclose(dw, numeric_dw, atol=1e-7)
assert np.allclose(db, numeric_db, atol=1e-7)
print('Производные проверены:', dw, db)

def fit_sigmoid(X, d, eta=0.5, epochs=700):
    w = np.random.default_rng(SEED).normal(0, 0.1, X.shape[1]); b = 0.
    history = []
    for epoch in range(epochs):
        dw, db = sigmoid_gradients(X, d, w, b)
        w -= eta * dw; b -= eta * db
        history.append(half_mse(sigmoid(X @ w+b), d))
    return w, b, np.array(history)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(z, step(z), label='Ступенька')
axes[0].plot(z, sigmoid(z), label='Сигмоида')
axes[0].set_xlabel('s'); axes[0].set_ylabel('y'); axes[0].legend(); axes[0].grid()
for eta in [0.05, 0.5, 5.0]:
    w, b, hist = fit_sigmoid(X_logic, d_and, eta=eta)
    axes[1].plot(hist, label=f'η={eta}')
    print(f'η={eta}: L={hist[-1]:.5f}; выходы={sigmoid(X_logic @ w+b)}')
axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('L'); axes[1].legend(); axes[1].grid()
plt.tight_layout(); plt.show()


Запустите готовые графики и объясните влияние η и геометрию ошибки.

In [ ]:
# Готовые демонстрации; писать код здесь не требуется.
# 1. Большой шаг на простой квадратичной функции: q(u)=u², градиент=2u.
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for eta in [0.05, 0.5, 1.2]:
    u = 1.; values = [u*u]
    for _ in range(15):
        u -= eta * 2*u; values.append(u*u)
    axes[0].semilogy(np.maximum(values, 1e-12), label=f'η={eta}')
axes[0].set_xlabel('Итерация'); axes[0].set_ylabel('q(u) (лог. шкала)')
axes[0].set_title('Шаг зависит от функции ошибки'); axes[0].legend(); axes[0].grid()
# 2. Двумерный срез ошибки сигмоидного нейрона: b фиксировано.
grid = np.linspace(-6, 6, 101)
W1, W2 = np.meshgrid(grid, grid)
scores = X_logic[:, 0, None, None]*W1 + X_logic[:, 1, None, None]*W2 - 1.
surface = 0.5*np.mean((sigmoid(scores)-d_and[:, None, None])**2, axis=0)
contours = axes[1].contourf(W1, W2, surface, levels=20)
fig.colorbar(contours, ax=axes[1], label='L')
axes[1].set_xlabel('w1'); axes[1].set_ylabel('w2'); axes[1].set_title('Срез поверхности: b = −1')
plt.tight_layout(); plt.show()


1. Почему σ удобна для градиентного обучения? Какие ограничения есть у насыщения?
2. От чего зависит L? Зачем множитель 1/2 и что меняется при его удалении?
3. Почему антиградиент уменьшает L только при подходящем шаге? Укажите диапазон η,
   приведенный в лекции, и поясните, почему это не универсальное ограничение.
4. Что обозначают «псевдопараболоид» и «гиперпсевдопараболоид» в лекции?
   В чем разница между срезом по двум параметрам и всей поверхностью?
5. Что получится, если убрать множитель y(1−y) из градиента сигмоидного нейрона?
6. Почему непосредственное градиентное обучение ступеньки не работает?
7. Дает ли переход к обобщенному правилу гарантию более быстрой сходимости?

**Ответ:** …


## F. XOR, скрытый слой и обратное распространение

Сеть **2 входа → 2 скрытых нейрона → 1 выход**, везде сигмоиды:
$$H=\sigma(XW_1+b_1),\qquad Y=\sigma(HW_2+b_2).$$
Для $D$ формы $(N,1)$ и $L=\frac1{2N}\sum_n(Y_n-D_n)^2$:
$$\delta_2=\frac{(Y-D)Y(1-Y)}N,\qquad
\delta_1=(\delta_2W_2^T)\odot H(1-H).$$
$$\nabla_{W_2}L=H^T\delta_2,\qquad \nabla_{W_1}L=X^T\delta_1.$$

Впишите только формулу δ1; весь остальной цикл уже подготовлен.
Обратите внимание: она использует веса выходного слоя **до** обновления.


In [ ]:
def hidden_delta(delta2, W2, H):
    # TODO: перенос градиента в скрытый слой и производная сигмоиды
    raise NotImplementedError('Допишите hidden_delta')


Запустите обучение XOR, сравните с одним нейроном в блоке C. Объясните, что идет вперед, что передается назад и кто делает шаг параметров.

In [ ]:
def fit_mlp(X, d, eta=1., epochs=12000, seed=SEED):
    D = np.asarray(d, dtype=float).reshape(-1, 1)
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (X.shape[1], 2)); b1 = np.zeros(2)
    W2 = rng.normal(0, 1, (2, 1)); b2 = np.zeros(1)
    history = []
    for epoch in range(epochs):
        H = sigmoid(X @ W1 + b1)
        Y = sigmoid(H @ W2 + b2)
        delta2 = (Y-D)*Y*(1-Y)/len(X)
        delta1 = hidden_delta(delta2, W2, H)
        # Все градиенты вычислены до изменения любых весов.
        gW2, gb2 = H.T @ delta2, delta2.sum(axis=0)
        gW1, gb1 = X.T @ delta1, delta1.sum(axis=0)
        W1 -= eta*gW1; b1 -= eta*gb1
        W2 -= eta*gW2; b2 -= eta*gb2
        if epoch % 100 == 0:
            fresh = sigmoid(sigmoid(X @ W1+b1) @ W2+b2)
            history.append((epoch+1, float(half_mse(fresh, D))))
    return (W1, b1, W2, b2), np.array(history)

params, history_mlp = fit_mlp(X_logic, d_xor)
W1, b1, W2, b2 = params
Y_xor = sigmoid(sigmoid(X_logic @ W1+b1) @ W2+b2).ravel()
print('XOR: вход / цель / выход / класс')
for x, d, y in zip(X_logic, d_xor, Y_xor):
    print(x.astype(int), d, round(float(y), 3), int(y >= 0.5))
assert np.array_equal((Y_xor >= 0.5).astype(int), d_xor)
plt.plot(history_mlp[:, 0], history_mlp[:, 1])
plt.xlabel('Эпоха'); plt.ylabel('L'); plt.title('XOR: сеть со скрытым слоем')
plt.grid(); plt.show()


1. Как скрытый слой меняет представление данных? … .
2. Почему backward идет от выхода к скрытому слою? … .
3. Какова роль η и где он используется? … .
4. Сопоставьте методы в таблице; отметьте возможности и ограничения:

| Метод | Использует целевой ответ d? | Сигмоида допустима? | Скрытые слои обучает для целевой ошибки? |
|---|---|---|---|
| Корреляционное правило Хебба Δw=ηxy | … | … | … |
| Пороговое правило персептрона Δw=η(d−y)x | … | … | … |
| Градиентное / обобщенное дельта-правило одного слоя | … | … | … |
| Backprop с градиентным шагом | … | … | … |

Преимущество и ограничение backprop по сравнению с предыдущими методами: … .


## G. Итоговый контроль

1. Напишите $s,y$ для порогового нейрона и связь $b,\theta$.
2. Для $x=(1,0),d=1,y=0,\eta=0.5$ найдите $\Delta w,\Delta b$; сформулируйте условие сходимости.
3. Назовите отличие проверки запоминания от проверки обобщения.
4. Напишите δ для сигмоиды и MSE; объясните, почему обычная ступенька не годится для прямого backprop.
5. Объясните роль скрытого слоя в XOR и роль обратного распространения.

**Ответы 1–5:** …

**Итог (2–3 предложения):** что удалось проверить экспериментом;
что стало причиной неуспеха одного нейрона; какие ограничения есть у выводов об обобщении? …

## Дополнительно

- Измените seed в многослойной сети: повторяется ли результат при том же лимите эпох?
- Сравните коррекцию сигмоидного нейрона с производной и без нее; не переносите
  свойства точного градиента на экспериментальную эвристику.
- Выполните один шаг backprop вручную для одного примера.
- Постройте поверхность L в 3D по тем же двум параметрам.
- Соберите отдельные наборы вариантов начертания цифр для обучения и теста;
  сравните обобщение с простым инвертированием пикселей.
